# DS-04 — Accessible Parking Locations, City of Melbourne

DS-04 supplies the *accessible parking* link of the access chain for the central city.
Unlike everything else in the register it needs no join: each feature is a designated
accessible bay carrying its own coordinate and a street segment description.

It replaces the City of Melbourne on-street car park bay restrictions and on-street
parking bays datasets, which were registered earlier in Iteration 1 and withdrawn on
31 August 2026. Those two could not be joined — the restrictions file held the
accessible designation but no location attribute of any kind, and six candidate routes
were tested and all failed, including the publisher's own documented bridge via the
parking bay sensors. Of the 102 bays designated for disabled parking there, 5 resolved
to a coordinate.

What this notebook has to settle:

1. **Grain and vocabulary.** Is every feature an accessible bay, or does the layer
   carry other categories? The publisher describes "over 500" spaces and the export
   returned more than that, so the difference has to be explained before any count is
   published.
2. **Identity.** The layer publishes no identifier, so a derived key is required and
   its stability has to be understood.
3. **Duplication.** Adjacent bays on the same segment sit metres apart. Are closely
   spaced points separate bays or repeated records?
4. **Geometry.** POINT Z with a zero altitude, in EPSG:4326.

Requires geopandas: `uv add geopandas`

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")

import pandas as pd
import profile_lib as pl

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

print("project root:", pl.PROJECT_ROOT)
print("raw zone:    ", pl.RAW_ROOT)


project root: C:\Users\nitin\Documents\Projects\Final_Project\SportAble
raw zone:     C:\Users\nitin\Documents\Projects\Final_Project\SportAble\_raw


In [2]:
try:
    import geopandas as gpd
    print("geopandas", gpd.__version__)
except ImportError:
    raise SystemExit("geopandas is required for this notebook. Run: uv add geopandas")

geopandas 1.1.4


In [3]:
raw = pl.resolve("DS-04")
p = pl.Profile(raw, "Accessible Parking Locations, City of Melbourne")
p.check("raw_integrity",
        "pass" if raw.sha_matches_manifest else ("info" if raw.sha_matches_manifest is None else "fail"),
        f"SHA-256 of the profiled object is {raw.sha256}", raw.sha256)
raw

RawObject(DS-04 dt=2026-08-31 CityofMelbourneaccessibledisabilityparking.kml 750,459B sha=5a7ea935ee36… [match])

## Load

The KML is read as-is. Google My Maps writes a folder per layer, so a multi-layer
document would need the layer named explicitly; a single-layer document reads
directly.

In [4]:
gdf = gpd.read_file(raw.path, driver="KML")
p.observe("feature_count", int(len(gdf)))
p.observe("declared_crs", str(gdf.crs))
p.observe("columns", list(gdf.columns))
p.observe("geometry_types", gdf.geom_type.value_counts())

p.check("crs_declared", "pass" if gdf.crs is not None else "fail",
        f"declared CRS is {gdf.crs}", str(gdf.crs))
p.contract("Reproject DS-04 to EPSG:7844 on load and record the source CRS in the source register.")

print(f"{len(gdf):,} features, CRS {gdf.crs}")
print(gdf.geom_type.value_counts().to_string())
gdf.head(3)

762 features, CRS EPSG:4326
Point    762


,id,Name,description,timestamp,begin,end,altitudeMode,tessellate,extrude,visibility,drawOrder,icon,description2,Latitude,Longitude,___UsageType,geometry
0,None,Wills Street between La Trobe Street and A'Bec...,﻿UsageType: Disable Parking\nLatitude: -37.810...,NaT,NaT,NaT,None,-1,0,-1,NaN,None,None,-37.8109733,144.9574159,Disable Parking,POINT Z (144.95742 -37.81097 0)
1,None,Wills Street between La Trobe Street and A'Bec...,﻿UsageType: Disable Parking\nLatitude: -37.810...,NaT,NaT,NaT,None,-1,0,-1,NaN,None,None,-37.8109313,144.9573964,Disable Parking,POINT Z (144.9574 -37.81093 0)
2,None,William Street between Little Bourke Street an...,﻿UsageType: Disable Parking\nLatitude: -37.814...,NaT,NaT,NaT,None,-1,0,-1,NaN,None,None,-37.8142479,144.9576576,Disable Parking,POINT Z (144.95766 -37.81425 0)


In [5]:
# Geometry arrives as POINT Z with a zero altitude, which is not a measurement.
# Carrying it into PostGIS would put a meaningless third dimension into every
# distance calculation.
has_z = bool(gdf.geometry.has_z.any())
p.observe("geometry_has_z", has_z)
if has_z:
    z_vals = gdf.geometry.apply(lambda g: g.z if g is not None and g.has_z else None)
    p.observe("z_value_distribution", z_vals.value_counts(dropna=False).head(5))
    p.check("geometry_dimension", "warn",
            f"geometry is 3D with altitude values {sorted(set(z_vals.dropna()))[:5]} — the Z is a "
            "placeholder, not an elevation measurement, and is dropped on load",
            has_z)
    p.contract("Drop the Z ordinate from DS-04 geometry on load. The altitude is a KML placeholder, not an elevation.")
else:
    p.check("geometry_dimension", "pass", "geometry is 2D", has_z)
has_z

True

## 1. Grain and vocabulary

The council's page describes over 500 designated on-street disability parking spaces.
If the export returns more features than that, either the layer carries categories
other than accessible bays, or it records adjacent bays individually. Both are fine;
guessing which is not.

In [6]:
USAGE_COL = "___UsageType"
if USAGE_COL not in gdf.columns:
    candidates = [c for c in gdf.columns if "usage" in c.lower() or "type" in c.lower()]
    p.check("usage_column_present", "warn",
            f"{USAGE_COL} is absent; candidates in the export are {candidates}", candidates)
    usage = None
else:
    usage = gdf[USAGE_COL]
    vc = usage.value_counts(dropna=False)
    p.observe("usage_type_vocabulary", vc)
    single = len(vc.dropna()) == 1
    p.check("usage_type_vocabulary", "pass" if single else "warn",
            (f"every feature carries the single category {vc.index[0]!r}, so the layer is "
             f"accessible bays only" if single else
             f"the layer carries {len(vc)} categories, so a filter is required before any "
             f"accessible parking count is published: {dict(vc)}"),
            vc)
    if not single:
        p.contract(f"Filter DS-04 on {USAGE_COL} before load. Only the accessible parking category becomes a facility row.")
    display(vc)

___UsageType
Disable Parking    762
Name: count, dtype: int64

In [7]:
# The publisher's own spelling is retained. A publisher's value is never silently
# corrected in place; normalisation happens downstream and is recorded.
if usage is not None:
    odd = [v for v in usage.dropna().unique() if "disable" in str(v).lower()]
    if odd:
        p.observe("publisher_spelling", odd)
        p.check("publisher_spelling", "info",
                f"the publisher writes {odd} rather than 'Disabled'. Retained verbatim in the raw "
                "zone and normalised downstream",
                odd)
        p.contract("Retain the publisher's UsageType spelling in the raw zone. Normalise on load and record the mapping.")
    print(odd)

['Disable Parking']


## 2. Identity

The `id` attribute is null on every feature, so the layer publishes no key. Without one
there is no stable reference for a bay and no publisher-side change detection between
fetches, which matters more than usual here because the endpoint offers no ETag or
Last-Modified either.

In [8]:
id_null = int(gdf["id"].isna().sum()) if "id" in gdf.columns else len(gdf)
p.observe("null_id_count", id_null)
p.check("published_identifier", "pass" if id_null == 0 else "warn",
        "the layer publishes an identifier on every feature" if id_null == 0
        else f"id is null on {id_null:,} of {len(gdf):,} features, so the layer publishes no key — "
             "a derived key is required and it is derived, not published",
        id_null)

name_col = "Name" if "Name" in gdf.columns else None
p.observe("name_column", name_col)
if name_col:
    p.observe("distinct_name_values", int(gdf[name_col].nunique()))
    p.observe("null_name_count", int(gdf[name_col].isna().sum()))
    p.observe("name_examples", gdf[name_col].dropna().head(3).tolist())
    p.check("street_description", "pass" if gdf[name_col].notna().all() else "warn",
            f"{gdf[name_col].nunique():,} distinct street segment descriptions across "
            f"{len(gdf):,} features"
            + ("" if gdf[name_col].notna().all() else f", {gdf[name_col].isna().sum():,} features have none"),
            int(gdf[name_col].nunique()))
gdf[name_col].dropna().head(5).tolist() if name_col else None

["Wills Street between La Trobe Street and A'Beckett Street",
 "Wills Street between La Trobe Street and A'Beckett Street",
 'William Street between Little Bourke Street and Lonsdale Street',
 'William Street between Little Bourke Street and Lonsdale Street',
 'William Street between Little Bourke Street and Lonsdale Street']

In [9]:
# Derive the key the transform will use, and check it actually distinguishes features.
COORD_PRECISION = 7
gdf["lat_r"] = gdf.geometry.y.round(COORD_PRECISION)
gdf["lon_r"] = gdf.geometry.x.round(COORD_PRECISION)
gdf["derived_key"] = [
    pl.synthetic_key(lat, lon, nm)
    for lat, lon, nm in zip(gdf["lat_r"], gdf["lon_r"], gdf[name_col] if name_col else [None] * len(gdf))
]

key_collisions = int(len(gdf) - gdf["derived_key"].nunique())
p.observe("derived_key_precision", COORD_PRECISION)
p.observe("derived_key_collisions", key_collisions)
p.check("derived_key_unique", "pass" if key_collisions == 0 else "warn",
        f"the derived key is unique across all {len(gdf):,} features"
        if key_collisions == 0 else
        f"{key_collisions:,} features share a derived key, meaning identical coordinate and street "
        "description — these are either duplicate records or genuinely co-located bays and must be "
        "resolved before load",
        key_collisions)
p.contract(
    f"DS-04 has no published identifier. Derive the key as sha256 of the coordinate rounded to "
    f"{COORD_PRECISION} decimal places plus the Name value, truncated to 16 hex characters. Record in "
    "the source card that the key is derived, and that it changes if the publisher moves a point."
)
p.limitation(
    "DS-04 publishes no identifier, so the primary key is derived from the coordinate and street "
    "description rather than supplied by the publisher. A moved point becomes a new key, and there is "
    "no way to distinguish a relocated bay from a removed bay plus a new one."
)
gdf["derived_key"].head(3).tolist()

['7c505c3795fc0256', 'cb387cfb2e08cda9', 'c54513be70b228c5']

## 3. Duplication

Adjacent bays on the same segment legitimately sit a few metres apart, so exact
coordinate duplicates and near-duplicates are different questions. Only the first is
evidence of a repeated record.

In [10]:
exact_dupes = int(gdf.duplicated(["lat_r", "lon_r"]).sum())
p.observe("exact_coordinate_duplicates", exact_dupes)
p.check("coordinate_duplicates", "pass" if exact_dupes == 0 else "warn",
        "no two features share an exact coordinate" if exact_dupes == 0
        else f"{exact_dupes:,} features share an exact coordinate with another feature — these are "
             "repeated records rather than adjacent bays and must be resolved before any count is "
             "published",
        exact_dupes)
if exact_dupes:
    display(gdf[gdf.duplicated(["lat_r", "lon_r"], keep=False)]
            .sort_values(["lat_r", "lon_r"])[[name_col, "lat_r", "lon_r"]].head(10))
exact_dupes

0

In [11]:
# How tightly are points clustered? A metric CRS is required for a distance in metres.
m = gdf.to_crs(7855)  # GDA2020 / MGA zone 55, metres, correct for Victoria
try:
    nearest = m.geometry.apply(lambda g: m.distance(g).nsmallest(2).iloc[-1])
    stats = {
        "median_nearest_neighbour_m": round(float(nearest.median()), 2),
        "min_nearest_neighbour_m": round(float(nearest.min()), 2),
        "within_1m": int((nearest < 1).sum()),
        "within_5m": int((nearest < 5).sum()),
        "within_20m": int((nearest < 20).sum()),
    }
    p.observe("nearest_neighbour_distances", stats)
    p.check("point_clustering", "info",
            f"median nearest neighbour is {stats['median_nearest_neighbour_m']} m; "
            f"{stats['within_1m']:,} features have a neighbour under 1 m and "
            f"{stats['within_5m']:,} under 5 m — sub-metre neighbours are worth inspecting, "
            "5 m apart is a normal adjacent bay",
            stats)
    display(pd.Series(stats))
except Exception as error:
    p.check("point_clustering", "info", f"nearest-neighbour scan not run: {error}", None)

median_nearest_neighbour_m      8.06
min_nearest_neighbour_m         0.48
within_1m                       2.00
within_5m                      73.00
within_20m                    455.00
dtype: float64

## 4. Coordinates and extent

In [12]:
flat = pd.DataFrame({"Latitude": gdf.geometry.y, "Longitude": gdf.geometry.x})
coord_stats = pl.check_coordinates(p, flat, "Latitude", "Longitude", label="features")

# The publisher extent is one council. Confirm the points actually sit inside it
# rather than trusting the label.
try:
    ref = pl.PROJECT_ROOT / "_reference" / "greater_melbourne_lga.gpkg"
    lga = gpd.read_file(ref)
    melb = lga[lga["lga_norm"] == "Melbourne"]
    inside = int(gdf.to_crs(melb.crs).within(melb.geometry.union_all()).sum())
    p.observe("features_inside_city_of_melbourne", inside)
    p.check("publisher_extent", "pass" if inside == len(gdf) else "warn",
            f"{inside:,} of {len(gdf):,} features fall inside the City of Melbourne boundary"
            + ("" if inside == len(gdf) else " — the remainder sit outside the stated publisher extent"),
            inside)
except Exception as error:
    p.check("publisher_extent", "info",
            f"extent check not run, run notebook 05 first to build the reference layer: {error}", None)

p.check("coverage_extent", "warn",
        "publisher scope is the City of Melbourne, one local government area of 31 — accessible "
        "on-street parking is no published information for the other 30, and that shortfall is a "
        "coverage figure for the quality report, never a gap to be filled from another source",
        {"lgas_covered": 1, "lgas_in_scope": 31})
p.limitation(
    "DS-04 covers the City of Melbourne only, one of the 31 Greater Melbourne councils. Accessible "
    "on-street parking is reported as no published information everywhere else. DS-01 records on-site "
    "accessible parking at venues across all 31 councils and remains the primary source for this link."
)
coord_stats

{'total': 762,
 'null_coords': 0,
 'out_of_range': 0,
 'likely_swapped_lat_lon': 0,
 'inside_gm_bbox': 762,
 'outside_gm_bbox': 0,
 'pct_inside_gm_bbox': 100.0}

## 5. What this source does not say

Recorded here so the limitations reach the venue card rather than being rediscovered
during Stage 3.

In [13]:
p.limitation(
    "The publisher states that disability parking locations on this map are subject to upgrade works "
    "under its Parking and Kerbside Management Plan. A point is where the council recorded a bay, not "
    "a guarantee that the bay is there today."
)
p.limitation(
    "A point is a bay, not a route to a bay. The layer records where a space is marked, not the kerb "
    "ramp, the crossing, the gradient or the path to a venue entrance. Proximity is a measured "
    "straight-line distance and the interface must not imply a step-free route."
)
p.limitation(
    "An on-street bay is not venue parking. Off-street accessible parking at a venue is recorded by "
    "DS-01 in Facility Features. The two are never merged into a single status without preserving "
    "which source said what."
)
p.contract("Set is_inside_venue false for every DS-04 row. The publisher records these as public on-street bays.")
p.contract("Ignore the KML description attribute. It duplicates the parsed fields as a text blob with a byte-order mark, and no attribute is inferred from it.")
p.contract("DS-04 has no publisher last-updated value and its endpoint offers no ETag or Last-Modified. Change detection is payload hash only, and the freshness note uses the retrieval date with that substitution recorded.")

for t in p.limitations:
    print("-", t, "\n")

- DS-04 publishes no identifier, so the primary key is derived from the coordinate and street description rather than supplied by the publisher. A moved point becomes a new key, and there is no way to distinguish a relocated bay from a removed bay plus a new one. 

- DS-04 covers the City of Melbourne only, one of the 31 Greater Melbourne councils. Accessible on-street parking is reported as no published information everywhere else. DS-01 records on-site accessible parking at venues across all 31 councils and remains the primary source for this link. 

- The publisher states that disability parking locations on this map are subject to upgrade works under its Parking and Kerbside Management Plan. A point is where the council recorded a bay, not a guarantee that the bay is there today. 

- A point is a bay, not a route to a bay. The layer records where a space is marked, not the kerb ramp, the crossing, the gradient or the path to a venue entrance. Proximity is a measured straight-line d

In [14]:
p.save()

DS-04 — Accessible Parking Locations, City of Melbourne
  object   CityofMelbourneaccessibledisabilityparking.kml  (750,459 bytes)
  dt       2026-08-31
  sha256   5a7ea935ee36b5d940bc28f7adf5e09097568b689911728fd6fedbdba923a6d1
  manifest hash matches

  Checks (WARN overall)
    [PASS] raw_integrity: SHA-256 of the profiled object is 5a7ea935ee36b5d940bc28f7adf5e09097568b689911728fd6fedbdba923a6d1
    [PASS] crs_declared: declared CRS is EPSG:4326
    [WARN] geometry_dimension: geometry is 3D with altitude values [0.0] — the Z is a placeholder, not an elevation measurement, and is dropped on load
    [PASS] usage_type_vocabulary: every feature carries the single category 'Disable Parking', so the layer is accessible bays only
    [INFO] publisher_spelling: the publisher writes ['Disable Parking'] rather than 'Disabled'. Retained verbatim in the raw zone and normalised downstream
    [WARN] published_identifier: id is null on 762 of 762 features, so the layer publishes no key — a deri

WindowsPath('C:/Users/nitin/Documents/Projects/Final_Project/SportAble/_profiles/dt=2026-08-31/DS-04.json')